In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [3]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

In [4]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("XGBoost_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

2025/07/01 18:19:58 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Regressor_HPO' does not exist. Creating a new experiment.


In [6]:
import json
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
param_list = {}

In [8]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y, y_pred, y_val = None, y_pred_val = None, verbose=True):
    # Calculate train metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    evs = explained_variance_score(y, y_pred)
    max_err = max_error(y, y_pred)
    medae = median_absolute_error(y, y_pred)
    metrics = {
        "mse": mse,
        "r2": r2,
        "mae": mae,
        "evs": evs,
        "max_err": max_err,
        "medae": medae
    }

    # If validation data is provided, calculate validation metrics
    if y_val is not None and y_pred_val is not None:
        # Calculate validation metrics
        mse_val = mean_squared_error(y_val, y_pred_val)
        r2_val = r2_score(y_val, y_pred_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)
        evs_val = explained_variance_score(y_val, y_pred_val)
        max_err_val = max_error(y_val, y_pred_val)
        medae_val = median_absolute_error(y_val, y_pred_val)
        metrics.update({
            "mse_val": mse_val,
            "r2_val": r2_val,
            "mae_val": mae_val,
            "evs_val": evs_val,
            "max_err_val": max_err_val,
            "medae_val": medae_val
        })

        # Create a table for formatting metrics
        if verbose:
            print("Metric       |    Train     |  Validation")
            print("-------------|--------------|---------------")
            print(f"MSE          |   {mse:.2e}   |   {mse_val:.2e}")
            print(f"R²           |   {r2:.4f}     |   {r2_val:.4f}")
            print(f"MAE          |   {mae:.2e}   |   {mae_val:.2e}")
            print(f"EVS          |   {evs:.4f}     |   {evs_val:.4f}")
            print(f"Max Error    |   {max_err:.2e}   |   {max_err_val:.2e}")
            print(f"Median AE    |   {medae:.2e}   |   {medae_val:.2e}")
            print()
            
    elif verbose:
        print(f"MSE          |   {mse:.2e}   ")
        print(f"R²           |   {r2:.4f}     ")
        print(f"MAE          |   {mae:.2e}   ")
        print(f"EVS          |   {evs:.4f}     ")
        print(f"Max Error    |   {max_err:.2e}   ")
        print(f"Median AE    |   {medae:.2e}   ")
        print()
    
    # Return metrics in a dictionary for reporting
    return metrics

## Preparacion de datos

In [12]:
df = pd.read_csv('../data/train/transformed_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14603 non-null  int64  
 1   Marca                    14603 non-null  object 
 2   Modelo                   14603 non-null  object 
 3   Versión                  14085 non-null  object 
 4   Título                   14603 non-null  object 
 5   Motor                    14439 non-null  float64
 6   Turbo                    14603 non-null  int64  
 7   cv                       7397 non-null   float64
 8   Tracción                 7197 non-null   object 
 9   Color                    14288 non-null  object 
 10  Tipo de combustible      14603 non-null  object 
 11  Puertas                  14603 non-null  float64
 12  Transmisión              14590 non-null  object 
 13  Con cámara de retroceso  3749 non-null   object 
 14  Kilómetros            

In [16]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Título", "Descripción", "Precio"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 61)
Val: (2921, 61)


## First Experiment

Ajuste de hiperparametros

In [ ]:
# Espacio de hiperparámetros refinado y más estable para MLPRegressor
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [1e-3, 1e-2],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [500, 1000]
}

# Modelo base con early stopping
mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, np.log1p(y_train))  # log-transform del target
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   6.51e+13   |   7.26e+13
R²           |   0.8690     |   0.8512
MAE          |   3.49e+06   |   3.93e+06
EVS          |   0.8690     |   0.8514
Max Error    |   4.78e+08   |   2.56e+08
Median AE    |   2.16e+06   |   2.34e+06

Best parameters found:
{
    "solver": "adam",
    "max_iter": 500,
    "learning_rate_init": 0.005,
    "learning_rate": "adaptive",
    "hidden_layer_sizes": [
        50,
        50
    ],
    "alpha": 0.01,
    "activation": "tanh"
}


In [19]:
# Espacio de hiperparámetros refinado según resultados anteriores
param_dist = {
    'hidden_layer_sizes': [(50,), (50, 50), (100, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.01, 0.005, 0.001],
    'learning_rate': ['adaptive', 'constant'],
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [500, 1000]
}

# Modelo base con early stopping
mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar con log-transform del target para estabilidad numérica
search.fit(X_train, np.log1p(y_train))

best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones (destransformar para que estén en escala original)
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["RandomizedSearchCV_MLP"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   6.51e+13   |   7.26e+13
R²           |   0.8690     |   0.8512
MAE          |   3.49e+06   |   3.93e+06
EVS          |   0.8690     |   0.8514
Max Error    |   4.78e+08   |   2.56e+08
Median AE    |   2.16e+06   |   2.34e+06

Best parameters found:
{
    "solver": "adam",
    "max_iter": 500,
    "learning_rate_init": 0.005,
    "learning_rate": "adaptive",
    "hidden_layer_sizes": [
        50,
        50
    ],
    "alpha": 0.01,
    "activation": "tanh"
}


In [22]:
def objective(trial):
    layer_size_str = trial.suggest_categorical('layer_size', ['50', '50_50', '100_50'])
    if layer_size_str == '50':
        hidden_layer_sizes = (50,)
    elif layer_size_str == '50_50':
        hidden_layer_sizes = (50, 50)
    else:
        hidden_layer_sizes = (100, 50)

    params = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': 'adam',
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-2, log=True),
        'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 0.001, 0.01, log=True),
        'max_iter': trial.suggest_int('max_iter', 500, 1000),
        'early_stopping': True,
        'validation_fraction': 0.1,
        'n_iter_no_change': 10,
        'random_state': 42
    }

    model = MLPRegressor(**params)
    scores = cross_val_score(
        model,
        X_train,
        np.log1p(y_train),
        cv=3,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    return -scores.mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=50)
best_params = study.best_params

# Ajustar mejor modelo con los mejores parámetros
best_params['early_stopping'] = True
best_params['validation_fraction'] = 0.1
best_params['n_iter_no_change'] = 10
best_params['solver'] = 'adam'
best_params['random_state'] = 42
# Supongamos que best_params tiene 'layer_size' en string
layer_size_str = best_params.pop('layer_size')

if layer_size_str == '50':
    best_params['hidden_layer_sizes'] = (50,)
elif layer_size_str == '50_50':
    best_params['hidden_layer_sizes'] = (50, 50)
elif layer_size_str == '100_50':
    best_params['hidden_layer_sizes'] = (100, 50)
else:
    raise ValueError(f"Unknown layer size string: {layer_size_str}")

best_model = MLPRegressor(**best_params)
best_model.fit(X_train, np.log1p(y_train))

# Predicciones y destransformación
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

# Reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["Optuna_MLP"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   7.18e+13   |   7.61e+13
R²           |   0.8556     |   0.8442
MAE          |   3.72e+06   |   4.10e+06
EVS          |   0.8559     |   0.8443
Max Error    |   4.92e+08   |   2.58e+08
Median AE    |   2.27e+06   |   2.39e+06

Best parameters found:
{
    "activation": "tanh",
    "alpha": 0.007374748049376304,
    "learning_rate": "constant",
    "learning_rate_init": 0.006509253592411294,
    "max_iter": 500,
    "early_stopping": true,
    "validation_fraction": 0.1,
    "n_iter_no_change": 10,
    "solver": "adam",
    "random_state": 42,
    "hidden_layer_sizes": [
        50,
        50
    ]
}


In [23]:
param_dist = {
    'hidden_layer_sizes': [(50,), (50, 50), (100, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [1e-3, 1e-2],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [500, 1000]
}

mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

halving_search = HalvingRandomSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Entrenamos con log-transform del target para mayor estabilidad
halving_search.fit(X_train, np.log1p(y_train))

best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

# Predicciones en escala original
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["HalvingRandomSearchCV_MLP"] = best_params

print("Best parameters found:")
print(best_params)


n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 7
min_resources_: 6
max_resources_: 11682
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 96
n_resources: 6
Fitting 3 folds for each of 96 candidates, totalling 288 fits


c:\Python312\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 96 is smaller than n_iter=1947. Running 96 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 32
n_resources: 18
Fitting 3 folds for each of 32 candidates, totalling 96 fits
----------
iter: 2
n_candidates: 11
n_resources: 54
Fitting 3 folds for each of 11 candidates, totalling 33 fits
----------
iter: 3
n_candidates: 4
n_resources: 162
Fitting 3 folds for each of 4 candidates, totalling 12 fits
----------
iter: 4
n_candidates: 2
n_resources: 486
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   7.35e+13   |   1.25e+14
R²           |   0.8521     |   0.7438
MAE          |   4.36e+06   |   5.06e+06
EVS          |   0.8524     |   0.7441
Max Error    |   3.49e+08   |   2.41e+08
Median AE    |   2.66e+06   |   3.04e+06

Best parameters found:
{'solver': 'adam', 'max_iter': 1000, 'learning_rate_init': 0.001, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (100, 50), 'alpha': 0.001, 'activation': 'relu'}


In [29]:
input_example = X_train.iloc[0:1]

for name, params in param_list.items():
    if 'early_stopping' in params:
        params['early_stopping'] = True
    if 'validation_fraction' in params:
        params['validation_fraction'] = 0.1
    if 'n_iter_no_change' in params:
        params['n_iter_no_change'] = 10
    if 'random_state' in params:
        params['random_state'] = 42
    
    model = MLPRegressor(**params)

    # Entrenamos con log-transform del target
    model.fit(X_train, np.log1p(y_train))

    # Predicciones y destransformación
    y_pred_train = np.expm1(model.predict(X_train))
    y_pred_val = np.expm1(model.predict(X_val))

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"MLP_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example, signature=signature)


2025/07/01 19:03:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run MLP_RandomizedSearchCV at: http://localhost:5000/#/experiments/841519844441403402/runs/50cb384af45c4570b9c8698a37029486
🧪 View experiment at: http://localhost:5000/#/experiments/841519844441403402


2025/07/01 19:04:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run MLP_RandomizedSearchCV_MLP at: http://localhost:5000/#/experiments/841519844441403402/runs/5e4368d3b7974124ba4c4e298768057b
🧪 View experiment at: http://localhost:5000/#/experiments/841519844441403402


2025/07/01 19:04:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run MLP_Optuna_MLP at: http://localhost:5000/#/experiments/841519844441403402/runs/4f6e7fa8c4704b94a028277c57a1c44e
🧪 View experiment at: http://localhost:5000/#/experiments/841519844441403402


2025/07/01 19:04:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

🏃 View run MLP_HalvingRandomSearchCV_MLP at: http://localhost:5000/#/experiments/841519844441403402/runs/e63ab029118942ecacdde912d8bcb448
🧪 View experiment at: http://localhost:5000/#/experiments/841519844441403402


In [32]:
# Hiperparámetros óptimos
params = {
    "solver": "adam",
    "max_iter": 1,  # solo un epoch por partial_fit
    "learning_rate_init": 0.005,
    "learning_rate": "adaptive",
    "hidden_layer_sizes": (50, 50),
    "alpha": 0.01,
    "activation": "tanh",
    "warm_start": True,  # necesario para entrenar con partial_fit
    "random_state": 42
}

# Preparar target transformado
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

# Inicializar modelo
model = MLPRegressor(**params)

# Iniciar MLflow
with mlflow.start_run(run_name="MLP_regresor_por_epoch"):

    mlflow.log_params(params)

    # Entrenamiento por epoch manual
    n_epochs = 100
    for epoch in range(n_epochs):
        model.fit(X_train, y_train_log)

        # Calcular loss (RMSE) en train y val (en escala log)
        y_train_pred_log = model.predict(X_train)
        y_val_pred_log = model.predict(X_val)

        rmse_train = np.sqrt(mean_squared_error(y_train_log, y_train_pred_log))
        rmse_val = np.sqrt(mean_squared_error(y_val_log, y_val_pred_log))

        mlflow.log_metric("train_log_rmse", rmse_train, step=epoch)
        mlflow.log_metric("val_log_rmse", rmse_val, step=epoch)

    # Predicciones finales en escala original
    y_train_pred = np.expm1(model.predict(X_train))
    y_val_pred = np.expm1(model.predict(X_val))

    # Métricas finales
    metrics = {
        "mse_train": mean_squared_error(y_train, y_train_pred),
        "mse_val": mean_squared_error(y_val, y_val_pred),
        "r2_train": r2_score(y_train, y_train_pred),
        "r2_val": r2_score(y_val, y_val_pred),
        "mae_train": mean_absolute_error(y_train, y_train_pred),
        "mae_val": mean_absolute_error(y_val, y_val_pred),
        "median_ae_val": median_absolute_error(y_val, y_val_pred),
        "evs_val": explained_variance_score(y_val, y_val_pred)
    }

    mlflow.log_metrics(metrics)

    # Guardar modelo
    mlflow.sklearn.log_model(model, "mlp_model")

print("Entrenamiento y log completo.")


c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: M

🏃 View run MLP_regresor_por_epoch at: http://localhost:5000/#/experiments/841519844441403402/runs/7bbc503fc202458a8f74682cb7b47cd1
🧪 View experiment at: http://localhost:5000/#/experiments/841519844441403402
Entrenamiento y log completo.
